# UGOT Robot — Basics Notebook

This notebook walks through the core features of the **UGOT robot**, including:
- Connecting to the robot and loading AI models
- Controlling lights and the display screen
- Moving and turning with the mecanum wheels
- Operating the mechanical arm and gripper
- Viewing the live camera feed
- Using AI perception: color, gesture, text, line-following, face, and AprilTag recognition

> **Before you start:** Make sure the robot is powered on and your computer is connected to the robot's Wi-Fi network (`192.168.88.1` by default). Run the cells **in order** from top to bottom.

## 1. Setup & Initialization

This cell does three things:

1. **Imports** the libraries we need - `ugot` for robot control, `numpy` and `cv2` (OpenCV) for image processing, and `PIL` / `IPython` for displaying images inside the notebook.
2. **Connects** to the robot over Wi-Fi at its default IP address (`192.168.88.1`).
3. **Loads the AI models** onto the robot so it can recognise colors, faces, gestures, text, lines, and AprilTags.

You should see `Done` printed when this cell finishes successfully.

In [ ]:
from ugot import ugot # pip install ugot
import numpy as np # pip install numpy
import cv2 # pip install opencv-python
from IPython.display import clear_output, Image
from PIL import Image as Image2 # pip install pillow
import time

# Create the robot object and connect to it over Wi-Fi.
# Change the IP address below if your robot uses a different address.
got = ugot.UGOT()
got.initialize("192.168.88.1")

# Load all AI perception models onto the robot.
# This may take a few seconds — the models run on the robot's own hardware.
got.load_models(
    [
        "word_recognition",  # Reads printed/written text (OCR)
        "color_recognition",  # Identifies colors in the camera frame
        "apriltag_qrcode",  # Detects AprilTag fiducial markers and QR codes
        "face_attribute",  # Detects faces and their attributes (e.g. age, expression)
        "face_recognition",  # Identifies specific registered people by face
        "line_recognition",  # Detects lines on the floor for line-following
        "gesture",  # Recognises hand gestures
    ]
)

# Select line sensor channel 0 (the default single-line tracker).
got.set_track_recognition_line(0)

# Start streaming video from the robot's camera.
got.open_camera()

print("Done")

## 2. Lights & Display Screen

The robot has four RGB LEDs (numbered 0–3) and a small colour screen.

- **`show_light_rgb`** — set any combination of lights to an RGB colour.
- **`turn_off_lights`** — turn all LEDs off.
- **`screen_display_background`** — fill the screen with a solid colour (0=Black, 1=White, 2=Purple, 3=Red, 4=Orange, 5=Yellow, 6=Green, 7=Cyan, 8=Blue).
- **`screen_print_text_newline`** — print a line of text on the screen in the chosen colour.
- **`screen_clear`** — erase the screen.

In [ ]:
# --- Lights ---
# Turn all four LEDs green for 1 second, then switch them off.
got.show_light_rgb(lights=[0, 1, 2, 3], red=0, green=255, blue=0)
time.sleep(1)
got.turn_off_lights()

# --- Display screen ---
# Colour codes: 0 Black; 1 White; 2 Purple; 3 Red; 4 Orange;
#               5 Yellow; 6 Green; 7 Cyan; 8 Blue
got.screen_display_background(color=0)  # Black background

# Print white text on the screen, then clear it after 1 second.
got.screen_print_text_newline(text="Hello!", color=1)
time.sleep(1)
got.screen_clear()

## 3. Movement

The UGOT uses **mecanum wheels**, which can move in any direction without turning first (holonomic drive). The key movement functions are:

| Function | What it does |
|---|---|
| `mecanum_translate_speed_times` | Slide in any direction (angle in degrees) |
| `mecanum_move_speed_times` | Move forward or backward |
| `mecanum_turn_speed_times` | Rotate left or right in place |
| `mecanum_stop` | Stop all wheel movement immediately |

**Direction codes:** `0` = forward, `1` = back, `2` = left, `3` = right  
**Unit codes:** `unit=1` means the `times` value is in **centimetres**; `unit=2` means **degrees**.

In [ ]:
# Direction reference: 0=forward, 1=back, 2=left, 3=right

# --- Strafe (translate) ---
# Slide forward (angle=0) at speed 30, for 50 cm.
# unit=1 means the 'times' value is in centimetres.
got.mecanum_translate_speed_times(angle=0, speed=30, times=50, unit=1)

# --- Forward / Backward ---
# Drive forward (direction=0) at speed 30, for 50 cm.
got.mecanum_move_speed_times(direction=0, speed=30, times=50, unit=1)

# --- Turning ---
# Turn left (turn=2) at speed 45, rotating 90 degrees.
# unit=2 means the 'times' value is in degrees.
got.mecanum_turn_speed_times(turn=2, speed=45, times=90, unit=2)

# --- Stop ---
# Immediately stop all wheel movement.
got.mecanum_stop()

## 4. Mechanical Arm & Gripper

The robot has a 3-joint arm with a gripper at the end.

- **`mechanical_arms_restory`** — return the arm to its home/rest position.
- **`mechanical_clamp_release`** — open the gripper.
- **`mechanical_clamp_close`** — close the gripper (used in later sections).
- **`mechanical_joint_control(angle1, angle2, angle3, duration)`** — move all three joints simultaneously to the given angles (in degrees) over the given duration (in milliseconds).

> **Tip:** Always add a `time.sleep()` after arm commands to give the motors time to reach their target position before issuing the next command.

In [ ]:
# --- Arm control ---

# Return the arm to its default rest/home position.
got.mechanical_arms_restory()
time.sleep(1)   # Wait for the arm to finish moving

# Open the gripper (release = open).
got.mechanical_clamp_release()
time.sleep(1)

# Open the gripper again (demonstrating the call — in a real program
# you would call mechanical_clamp_close() here to close it).
got.mechanical_clamp_release()
time.sleep(1)

# Move the arm joints to specific angles.
# angle1 = base rotation, angle2 = shoulder, angle3 = elbow (all in degrees)
# duration = time in milliseconds to reach the target pose
got.mechanical_joint_control(angle1=0, angle2=45, angle3=45, duration=500)
time.sleep(1)   # Wait for the arm to reach the target pose

## 5. Live Camera Feed

These two cells set up and run a live camera preview inside the notebook.

**How it works:**
1. `got.read_camera_data()` fetches the latest JPEG frame from the robot as raw bytes.
2. `np.frombuffer` + `cv2.imdecode` converts those bytes into a NumPy image array (the format OpenCV uses).
3. `cv2.cvtColor` converts from BGR (OpenCV's default channel order) to RGB (what PIL and most display tools expect).
4. `display()` + `clear_output(wait=True)` renders the frame in the notebook and replaces the previous one, creating a smooth live-view effect.

**To stop the loop:** press the **Stop** button in the Jupyter toolbar, or press **interrupt kernel**.

In [ ]:
def display_frame():
    """Grab one camera frame and display it inline in the notebook."""
    frame = got.read_camera_data()
    if frame is not None:  # Checks if the frame is real!
        nparr = np.frombuffer(frame, np.uint8)  # Wrap raw bytes as a NumPy array
        data = cv2.imdecode(nparr, cv2.IMREAD_COLOR)  # Decode JPEG → BGR image array
        frame_rgb = cv2.cvtColor(
            data, cv2.COLOR_BGR2RGB
        )  # Convert BGR → RGB for display

        display(Image2.fromarray(frame_rgb))  # Render frame in the notebook
        clear_output(wait=True)  # Replace the previous frame (live-view effect)

In [ ]:
# Run the camera preview in a continuous loop.
# Press Stop in the toolbar or interrupt the kernel to exit.
try:
    while True:
        display_frame()

except KeyboardInterrupt:
    print("Done!")

## 6. Color Recognition

The robot's camera can identify colors in real time using the `color_recognition` model loaded during setup.

- **`get_color_total_info()`** returns a list of detected color entries. Each entry contains information about a color region found in the frame (e.g. the color name and its position/size).
- We take index `[0]` to inspect the most prominent detection.

The loop prints the color data continuously until you stop it. **To stop:** interrupt the kernel or press the Stop button — this also stops the robot's wheels as a safety measure.

In [ ]:
try:
    print("Starting color recognition...")
    while True:
        # Read the latest color detection from the robot's camera.
        # [0] selects the first (most prominent) detected color region.
        color = got.get_color_total_info()[0]

        print(color)
        clear_output(wait=True)  # Overwrite the previous line for a cleaner display

except KeyboardInterrupt:
    got.mecanum_stop()   # Safety stop — halt the wheels before exiting
    print("Done!")

## 7. Gesture Recognition

The `gesture` model can recognise hand gestures (e.g. thumbs up, peace sign, open palm) shown in front of the camera.

- **`get_gesture_result()`** returns the currently detected gesture as a string or structured result.

Hold a gesture in front of the camera and watch the output update. **To stop:** interrupt the kernel.

In [ ]:
try:
    print("Starting gesture recognition...")
    while True:
        # Read the current gesture detected by the camera.
        gesture = got.get_gesture_result()

        print(gesture)
        clear_output(wait=True)

except KeyboardInterrupt:
    got.mecanum_stop()
    print("Done!")

## 8. Text Recognition (OCR)

The `word_recognition` model lets the robot read printed text using its camera — useful for reading name tags, signs, or labels.

- **`get_words_result()`** returns any text currently visible in the camera frame as a string.

Hold a piece of text up to the camera to see it recognised. **To stop:** interrupt the kernel.

In [ ]:
try:
    print("Starting text recognition...")
    while True:
        # Read any text currently visible in the camera frame.
        text = got.get_words_result()

        print("Text:", text)

        clear_output(wait=True)

except KeyboardInterrupt:
    got.mecanum_stop()
    print("Done!")

## 9. Line Following

The `line_recognition` model lets the robot follow a line (e.g. a black tape line on the floor) by continuously reading the line's position and adjusting its steering.

**How `line_follow` works:**
1. It reads the **offset** — how far left or right the line is from the robot's centre.
2. It converts the offset into a **rotation speed** using a multiplier (`mult`). A larger offset means a sharper correction.
3. It drives forward at the given `speed` while rotating to re-centre on the line (`mecanum_move_xyz`).
4. It returns the detected line type and position coordinates so the calling program can make higher-level decisions (e.g. stop at an intersection).

> **To use this:** call `line_follow()` in a loop from your own code. The function itself doesn't loop — it performs one step of correction per call.

In [ ]:
def line_follow(mult=0.25, speed=35):
    """Follow the detected line by turning proportionally to the line offset.
    
    Args:
        mult  (float): Steering sensitivity. Higher values = sharper turns. Default 0.25.
        speed (int):   Forward drive speed (cm/s). Default 35.
    
    Returns:
        tuple: (line_type, x, y) — the detected line pattern and its position coordinates.
    """
    # Read line-tracking information from the robot.
    # offset    — how far the line is from the robot's centre (negative=left, positive=right)
    # line_type — the pattern detected (straight, corner, T-junction, etc.)
    # x, y      — pixel coordinates of the detected line
    offset, line_type, x, y = got.get_single_track_total_info()

    # Scale the offset into a rotation speed.
    # Larger offset -> stronger rotation to re-centre on the line.
    rotation_speed = int(offset * mult)

    # Drive forward while rotating to stay aligned with the line.
    # mecanum_move_xyz(x_speed, y_speed, z_speed): x=strafe, y=forward, z=rotation
    got.mecanum_move_xyz(x_speed=0, y_speed=speed, z_speed=rotation_speed)

    # Return detection info so the caller can act on intersections or endpoints.
    return line_type, x, y

In [ ]:
try:
    while True:
        line_follow()
        
except KeyboardInterrupt:
    got.mecanum_stop()

## 10. Face Recognition

The robot can learn and remember specific faces. This section covers:

1. **Registering** a face with a name
2. **Listing** all registered faces
3. **Deleting** a single face or all faces
4. **Live preview** with face + text overlays
5. **Autonomous search-and-approach** — the robot spins to find a named person, then drives toward them

> **Note:** Faces are stored persistently on the robot. You only need to register a face once; it will be remembered across sessions.

In [ ]:
# Register a new face with the given name.
# Point the camera at the person's face when running this cell.
got.face_recognition_add_name(name="Bob")

In [ ]:
# List all face names currently stored on the robot.
got.face_recognition_get_all_names()

In [ ]:
# Remove a single registered face by name.
got.face_recognition_delete_name(name="Bob")

In [ ]:
# Remove ALL registered faces from the robot's memory.
faces = got.face_recognition_get_all_names()

for face in faces:
    got.face_recognition_delete_name(name=face)

In [ ]:
def face_text_preview():
    """Display a live camera frame annotated with detected text and face data."""
    # Grab the latest camera frame as raw bytes
    frame = got.read_camera_data()

    # Decode the bytes into an OpenCV image array
    nparr = np.frombuffer(frame, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    # Overlay any detected text in the top-left corner of the frame
    name = got.get_words_result()
    if name:
        cv2.putText(img, f"text: {name}", (20, 40), 0, 0.8, (0, 255, 0), 2)

    # Overlay face recognition data if any faces are detected.
    # Each face entry is: [name, center_x, center_y, height]
    faces = got.get_face_recognition_total_info()
    if faces:
        face = faces[0]  # Only look at the first detected face
        cv2.putText(img, f"face: {face}", (20, 70), 0, 0.8, (0, 255, 0), 2)

    # Encode the annotated frame as JPEG and display it inline in the notebook
    _, jpeg = cv2.imencode(".jpg", img)
    clear_output(wait=True)
    display(Image(data=jpeg.tobytes()))

In [ ]:
# Open the camera and run the annotated face/text preview in a continuous loop.
# The live feed will show bounding boxes and labels for detected faces and text.
# To stop: press the Stop button or interrupt the kernel (I, I).
got.open_camera()

try:
    while True:
        face_text_preview()

except KeyboardInterrupt:
    print("Done")
    got.mecanum_stop()
    clear_output(wait=True)

In [ ]:
def face_find_and_approach(
    gap=10, target_name="Stranger", turn_spd=15, strafe_spd=10, fwd_spd=10, height=80, adjust_turn=10
):
    """
    Spin until the target person is found, then drive toward them.

    Phase 1 — Search:
        The robot turns continuously, checking each frame for the target's
        face (via face_recognition) or name tag (via OCR). When found, it
        stops and does a small corrective turn to face them, then moves to Phase 2.

    Phase 2 — Approach:
        The robot drives toward the face, strafing left/right to keep it
        centered in frame, until the face appears large enough (close enough).

    Args:
        gap          (int):   Pixel dead-zone around the frame centre before strafing. Default 10.
        target_name  (str):   Name of the person to find (must match a registered face or OCR text).
        turn_spd     (int):   Rotation speed during the search phase. Default 15.
        strafe_spd   (int):   Side-to-side correction speed during approach. Default 10.
        fwd_spd      (int):   Forward speed during approach. Default 10.
        height       (int):   Stop when the face bounding-box height exceeds this value (i.e. close enough). Default 80.
        adjust_turn  (int):   Small corrective rotation (degree-units) after spotting the target. Default 10.
    """
    face_name = None  # Will hold the name from face recognition once a face is found

    try:
        # ---- Phase 1: Spin and search ----
        while True:
            # Turn clockwise slowly to scan the environment
            got.mecanum_turn_speed(turn=3, speed=turn_spd)

            # Read text visible in the frame (e.g. a printed name tag)
            name = got.get_words_result()

            # Check for any recognised faces in the frame
            faces = got.get_face_recognition_total_info()
            if faces:
                face_name = faces[0][0]  # faces[0] = first face; [0] = its name string

            # If either the OCR text or the face name matches our target, we found them!
            if name == target_name or face_name == target_name:
                got.mecanum_stop()
                print(f"Saw {target_name}!")

                # Small corrective clockwise turn to better centre the robot on the target
                got.mecanum_turn_speed_times(turn=3, speed=20, times=adjust_turn, unit=2)
                break  # Exit Phase 1, move on to Phase 2

        # ---- Phase 2: Approach the target ----
        while True:
            name = got.get_words_result()
            faces = got.get_face_recognition_total_info()

            if not faces:
                # Lost the face — inch forward slowly to try to find it again
                got.mecanum_translate_speed(angle=0, speed=fwd_spd)
            else:
                c_x = faces[0][1]  # Horizontal centre of the face in the frame (0–640 px)
                h   = faces[0][3]  # Height of the face bounding box — proxy for distance
                if h < height:     # Still far away — keep approaching
                    if c_x < 320 - gap:
                        # Face is to the LEFT of centre — strafe left while moving forward
                        got.mecanum_move_xyz(
                            x_speed=-strafe_spd, y_speed=fwd_spd, z_speed=0
                        )
                    elif c_x > 320 + gap:
                        # Face is to the RIGHT of centre — strafe right while moving forward
                        got.mecanum_move_xyz(x_speed=strafe_spd, y_speed=fwd_spd, z_speed=0)
                    else:
                        # Face is centred but still far away — drive straight forward
                        got.mecanum_move_xyz(x_speed=0, y_speed=fwd_spd, z_speed=0)
                else:
                    # Face is centred AND large enough — we've arrived!
                    got.mecanum_stop()
                    print(f"Reached {name}!")
                    break

        got.mecanum_stop()

    except KeyboardInterrupt:
        print("Done")
        got.mecanum_stop()

In [ ]:
face_find_and_approach(
    gap=10,
    target_name="Stranger",
    turn_spd=15,
    strafe_spd=10,
    fwd_spd=10,
    height=80,
    adjust_turn=10,
)

## 11. AprilTag Recognition

[AprilTags](https://april.eecs.umich.edu/software/apriltag) are square fiducial markers (like QR codes) that the robot can detect with high accuracy. Each tag has a unique **ID** and the robot can estimate its **distance** and **position** in the camera frame.

This section provides:

- **`AP_centralization_approaching`** — drive toward a detected tag, strafing to keep it centred, until the robot is within a target distance.
- **`pick_up`** — use the arm to pick up the object the tag is attached to, using the tag's position to calculate the correct joint angles.
- **Live preview** — draw a bounding box around detected tags in real time.
- **Full pick-up sequence** — scan for a tag, approach it, and pick it up.

**AprilTag data fields** (from `get_apriltag_total_info()`):

| Index | Meaning |
|---|---|
| `[0]` | Tag ID |
| `[1]` | Centre X (pixels, 0–640) |
| `[2]` | Centre Y (pixels) |
| `[3]` | Height (pixels) |
| `[4]` | Width (pixels) |
| `[6]` | Estimated distance (metres) |

In [ ]:
def AP_centralization_approaching(distance=0.15, gap=20, fwd_spd=10, strafe_spd=10):
    """
    Drive toward a detected AprilTag, keeping it centred in the camera frame.

    Parameters:
        distance   (float): Stop when the tag is within this many metres (default 0.15 m).
        gap        (int):   Pixel tolerance around centre (320 px) before strafing (default 20 px).
        fwd_spd    (int):   Forward drive speed (default 10).
        strafe_spd (int):   Left/right correction speed (default 10).
    """
    try:
        # Get an initial reading to confirm a tag is visible before entering the loop.
        AP_info = got.get_apriltag_total_info()
        AP_x        = AP_info[0][1]  # Horizontal pixel position of the tag (0=left, 640=right)
        AP_distance = AP_info[0][6]  # Estimated distance to the tag in metres

        while True:
            # Refresh tag data every iteration for responsive corrections.
            AP_info     = got.get_apriltag_total_info()
            AP_x        = AP_info[0][1]
            AP_distance = AP_info[0][6]

            if AP_x < 320 - gap:
                # Tag is to the LEFT of centre — strafe left to re-align.
                # mecanum_move_xyz(x, y, z): x=strafe, y=forward, z=rotation
                got.mecanum_move_xyz(-strafe_spd, strafe_spd, 0)
            elif AP_x > 320 + gap:
                # Tag is to the RIGHT of centre — strafe right to re-align.
                got.mecanum_move_xyz(strafe_spd, strafe_spd, 0)
            elif AP_distance > distance:
                # Tag is centred but still too far — drive straight forward.
                got.mecanum_move_xyz(0, fwd_spd, 0)
            else:
                # Tag is centred AND within target distance — stop and exit.
                got.mecanum_stop()
                print("It's too close, let's stop.")
                break
    except IndexError:
        # IndexError means get_apriltag_total_info() returned an empty list — no tag visible.
        print("ERROR: AprilTag cannot be seen.")


def pick_up():
    """
    Pick up the object identified by the closest visible AprilTag.

    Assumes the robot is already aligned and close to the target
    (i.e., AP_centralization_approaching() has just completed).
    """
    # Read the tag's current position and distance for arm targeting.
    AP_info = got.get_apriltag_total_info()
    try:
        AP_x        = AP_info[0][1]
        AP_distance = AP_info[0][6]

        # Move arm to a neutral ready position and open the gripper.
        # joint_control(j1, j2, j3, duration_ms): j2=30, j3=30 tilts the arm slightly forward.
        got.mechanical_joint_control(0, 30, 30, 1000)
        got.mechanical_clamp_release()   # Open gripper before extending arm
        time.sleep(2)                    # Wait for the gripper to fully open

        # Calculate arm joint angles based on the tag's camera position.
        # joint1 (base rotation): convert pixel offset from centre to degrees.
        #   The negative factor compensates for the camera being mirrored horizontally.
        joint1 = int((AP_x - 320) * -1 / 10)

        # joint3 (elbow extension): convert distance (m) to an extension angle.
        #   The -80 offset accounts for the arm's resting angle calibration.
        joint3 = int(AP_distance * 100 - 80)

        # Move arm to the computed pick-up pose.
        got.mechanical_joint_control(joint1, 0, joint3, 500)
        print(f"Joint1 value is: {joint1}, Joint3 value is: {joint3}.")
        time.sleep(1)  # Wait for the arm to reach the target pose

        # Close the gripper to grasp the object, then lift the arm to a carry position.
        got.mechanical_clamp_close()
        time.sleep(2)  # Wait for the gripper to fully close before lifting
        got.mechanical_joint_control(0, 30, 30, 1000)  # Return arm to neutral carry pose
    except IndexError:
        print("ERROR: AprilTag cannot be seen.")

In [ ]:
# Live AprilTag preview — draws a bounding box and distance label around any detected tag.
# Run this cell to verify the robot can see your AprilTag before running the pick-up sequence.
# To stop: interrupt the kernel (I, I).
got.open_camera()
try:
    while True:
        # Grab the latest camera frame as raw bytes
        frame = got.read_camera_data()
        # Decode the bytes into an OpenCV image array
        nparr = np.frombuffer(frame, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        tags = got.get_apriltag_total_info()
        if tags:
            tag = tags[0]  # Only annotate the first detected tag

            # Extract centre position and dimensions from tag data
            cx, cy = int(tag[1]), int(tag[2])
            half_w, half_h = int(tag[4] / 2), int(tag[3] / 2)

            # Compute bounding box corners from centre + half-dimensions
            x1, y1 = cx - half_w, cy - half_h
            x2, y2 = cx + half_w, cy + half_h

            # Draw a green bounding box and a red centre dot
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.circle(img, (cx, cy), 4, (0, 0, 255), -1)

            # Label the tag with its ID and estimated distance
            label = f"ID: {tag[0]}  dist: {tag[6]:.1f}cm"
            cv2.putText(
                img, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2
            )

        # Encode the annotated frame as JPEG and display it inline in the notebook
        _, jpeg = cv2.imencode(".jpg", img)
        clear_output(wait=True)
        display(Image(data=jpeg.tobytes()))

except KeyboardInterrupt:
    print("Done")
    got.mecanum_stop()

In [ ]:
# Full autonomous pick-up sequence:
#   1. The robot drives forward slowly, scanning for an AprilTag.
#   2. When a tag is spotted, it centres on the tag and approaches to ~14 cm.
#   3. It then uses the arm to pick up the tagged object.
# To stop at any time: interrupt the kernel (I, I).
try:
    print("Starting AprilTag recognition...")
    while True:
        AP_info = got.get_apriltag_total_info()

        if AP_info:
            # Tag found — approach it and pick it up.
            AP_centralization_approaching(
                distance=0.14, gap=18, fwd_spd=11, strafe_spd=5
            )
            pick_up()
            time.sleep(2)
            break  # Exit after a successful pick-up
        else:
            # No tag visible yet — creep forward to search.
            got.mecanum_move_speed(0, 15)

except KeyboardInterrupt:
    got.mecanum_stop()
    print("Done!")